# Figure S8: PCS parameter and initiation-time robustness

This notebook evaluates PCS robustness to hard thresholds, terminal-window length, early-reference duration, and circular endpoint perturbations.

## Setup and analysis scope

Baseline: S > 2.0, R < 1.0, a 1 h terminal window, and a 12 h early reference. Positive and negative endpoint offsets use the same circular label-shift convention as the existing PCS null test.

In [ ]:
from pathlib import Path
import importlib.util
import re
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PAPER_DIR = Path(r"D:\Pycharm_Project\Find_precursor_signs\ILLgarben\uilt_oneday_all\uilt_paper_figure")
PROJECT_ROOT = PAPER_DIR.parents[1]
PCS_SCRIPT = PROJECT_ROOT / "uilt_oneday_all" / "03visual" / "09_export_precursor_curve_metrics.py"
DEC_LABEL_DIR = PROJECT_ROOT / "uilt_oneday_all" / "02clustering" / "result" / "DEC_result" / "all_parement"
OUT_DIR = PAPER_DIR / "Figure_S8_PCS_robustness_exports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
spec = importlib.util.spec_from_file_location("pcs_reference", PCS_SCRIPT)
if spec is None or spec.loader is None:
    raise ImportError(f"Cannot import PCS reference script: {PCS_SCRIPT}")
pcs = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pcs)
DT_SECONDS = int(pcs.DT_SECONDS)
BASELINE = {"z_threshold": 2.0, "c_threshold": 1.0, "terminal_hours": 1.0, "early_hours": 12.0}
NULL_SHIFTS_PER_EVENT = 40
NULL_REPLICATES = 2000
NULL_SEED = 20260817


## Load event labels and compute sensitivity results

In [ ]:
def event_number(event_name):
    match = re.search(r"(\d+)$", str(event_name))
    return int(match.group(1)) if match else 10**9

event_files = sorted([(p.name.split("_")[0], p) for p in DEC_LABEL_DIR.glob("event*_all_parement.npz")], key=lambda item: event_number(item[0]))
if len(event_files) != 44:
    raise ValueError(f"Expected 44 event files, found {len(event_files)}")
labels_by_event = {}
for event_name, event_path in event_files:
    with np.load(event_path, allow_pickle=True) as archive:
        labels_by_event[event_name] = np.asarray(archive["y_pred_last"], dtype=int)
if len({len(values) for values in labels_by_event.values()}) != 1:
    raise ValueError("Event label sequences do not have equal duration")

def compute_metrics(labels, early_hours=12.0, terminal_hours=1.0):
    time_points = np.arange(len(labels), dtype=float) * DT_SECONDS
    metrics = pcs.compute_cluster_metrics(time_points=time_points, time_labels=labels, unique_labels=np.unique(labels), dt=DT_SECONDS, bg_ratio=float(early_hours / 24.0)).copy()
    terminal_seconds = int(round(float(terminal_hours) * 3600.0))
    for cluster_id in metrics["cluster"].astype(int):
        terminal = pcs.compute_terminal_growth_metrics(mask=(labels == cluster_id).astype(int), dt=DT_SECONDS, terminal_window_s=terminal_seconds)
        row_mask = metrics["cluster"].eq(cluster_id)
        for column, value in terminal.items():
            metrics.loc[row_mask, column] = value
    return metrics

def select_state(labels, event_name, z_threshold=2.0, c_threshold=1.0, terminal_hours=1.0, early_hours=12.0):
    metrics = compute_metrics(labels, early_hours=early_hours, terminal_hours=terminal_hours)
    valid = metrics.dropna(subset=["z_final", "rate_acceleration", "c_value", "terminal_growth_ratio"]).copy()
    result = {"event": event_name, "selected": False, "cluster": -1, "pcs": np.nan, "terminal_enrichment": np.nan}
    if valid.empty:
        return result
    for column in ["z_final", "rate_acceleration", "c_value", "terminal_growth_ratio"]:
        valid[f"z_{column}"] = pcs.within_event_zscore(valid[column])
    valid["pcs"] = valid["z_z_final"] + valid["z_rate_acceleration"] - valid["z_c_value"] + valid["z_terminal_growth_ratio"]
    candidates = valid[(valid["z_final"] > float(z_threshold)) & (valid["c_value"] < float(c_threshold)) & (valid["terminal_growth_flag"] == 1)].copy()
    if candidates.empty:
        return result
    row = candidates.sort_values(["pcs", "z_final", "terminal_growth_ratio", "sample_count"], ascending=[False, False, False, False]).iloc[0]
    cluster_id = int(row["cluster"])
    mask = (labels == cluster_id).astype(float)
    terminal_points = max(int(round(float(terminal_hours) * 3600.0 / DT_SECONDS)), 1)
    result.update({"selected": True, "cluster": cluster_id, "pcs": float(row["pcs"]), "terminal_enrichment": float(mask[-terminal_points:].mean() / mask.mean()) if mask.mean() > 0 else np.nan})
    return result

def observed_summary(config, labels_override=None):
    labels_override = labels_by_event if labels_override is None else labels_override
    rows = [select_state(labels, event, **config) for event, labels in labels_override.items()]
    values = np.asarray([row["terminal_enrichment"] for row in rows if row["selected"]], dtype=float)
    return {"rows": rows, "selected_count": int(sum(row["selected"] for row in rows)), "median_enrichment": float(np.nanmedian(values)) if len(values) else np.nan}

def null_summary(config, seed_offset=0):
    rng = np.random.default_rng(NULL_SEED + int(seed_offset))
    event_enrichment, event_selected = [], []
    for event, labels in labels_by_event.items():
        shifts = rng.integers(1, len(labels), size=NULL_SHIFTS_PER_EVENT)
        enrichments, selected = [], []
        for shift in shifts:
            row = select_state(np.roll(labels, int(shift)), event, **config)
            enrichments.append(row["terminal_enrichment"]); selected.append(row["selected"])
        event_enrichment.append(np.asarray(enrichments, dtype=float)); event_selected.append(np.asarray(selected, dtype=bool))
    rep_rng = np.random.default_rng(NULL_SEED + 100000 + int(seed_offset))
    medians = np.full(NULL_REPLICATES, np.nan, dtype=float); counts = np.zeros(NULL_REPLICATES, dtype=int)
    for replicate in range(NULL_REPLICATES):
        values = []
        for enrichments, selected in zip(event_enrichment, event_selected):
            index = int(rep_rng.integers(0, len(enrichments)))
            if selected[index] and np.isfinite(enrichments[index]): values.append(enrichments[index])
        counts[replicate] = len(values)
        if values: medians[replicate] = float(np.nanmedian(values))
    return {"median_enrichment": medians, "selected_count": counts}

def run_config(config, seed_offset=0, with_null=True):
    observed = observed_summary(config)
    output = {"observed": observed, "null": None, "null_low": np.nan, "null_high": np.nan, "null_count_low": np.nan, "null_count_high": np.nan}
    if with_null:
        null = null_summary(config, seed_offset=seed_offset); output["null"] = null
        valid = null["median_enrichment"][np.isfinite(null["median_enrichment"])]
        if len(valid): output["null_low"], output["null_high"] = np.nanpercentile(valid, [2.5, 97.5])
        output["null_count_low"], output["null_count_high"] = np.percentile(null["selected_count"], [2.5, 97.5])
    return output

parameter_configs = [
    ("Cumulative anomaly", "S > 1.5", {**BASELINE, "z_threshold": 1.5}), ("Cumulative anomaly", "S > 2.0", {**BASELINE, "z_threshold": 2.0}), ("Cumulative anomaly", "S > 2.5", {**BASELINE, "z_threshold": 2.5}),
    ("Nonlinear condition", "R < 0.8", {**BASELINE, "c_threshold": 0.8}), ("Nonlinear condition", "R < 1.0", {**BASELINE, "c_threshold": 1.0}), ("Nonlinear condition", "R < 1.2", {**BASELINE, "c_threshold": 1.2}),
    ("Terminal window", "0.5 h", {**BASELINE, "terminal_hours": 0.5}), ("Terminal window", "1 h", {**BASELINE, "terminal_hours": 1.0}), ("Terminal window", "2 h", {**BASELINE, "terminal_hours": 2.0}),
    ("Early reference", "6 h", {**BASELINE, "early_hours": 6.0}), ("Early reference", "9 h", {**BASELINE, "early_hours": 9.0}), ("Early reference", "12 h", {**BASELINE, "early_hours": 12.0}),
]
parameter_results, parameter_rows = {}, []
for index, (family, label, config) in enumerate(parameter_configs):
    result = run_config(config, seed_offset=1000 + index * 97, with_null=True); parameter_results[(family, label)] = result
    parameter_rows.append({"family": family, "parameter": label, **config, "observed_selected_events": result["observed"]["selected_count"], "observed_median_enrichment": result["observed"]["median_enrichment"], "null_enrichment_2.5%": result["null_low"], "null_enrichment_97.5%": result["null_high"], "null_selected_events_2.5%": result["null_count_low"], "null_selected_events_97.5%": result["null_count_high"]})
parameter_summary = pd.DataFrame(parameter_rows)

heatmap_rows = []
for z_threshold in [1.5, 2.0, 2.5]:
    for c_threshold in [0.8, 1.0, 1.2]:
        result = observed_summary({**BASELINE, "z_threshold": z_threshold, "c_threshold": c_threshold})
        heatmap_rows.append({"z_threshold": z_threshold, "c_threshold": c_threshold, "selected_events": result["selected_count"], "median_enrichment": result["median_enrichment"]})
heatmap_summary = pd.DataFrame(heatmap_rows)

offsets_min = np.array([-30, -20, -10, -5, 0, 5, 10, 20, 30], dtype=int); time_rows = []
for offset in offsets_min:
    shifted = {event: np.roll(labels, int(round(offset * 60 / DT_SECONDS))) for event, labels in labels_by_event.items()}
    result = observed_summary(BASELINE, labels_override=shifted)
    time_rows.append({"offset_min": int(offset), "selected_events": result["selected_count"], "median_enrichment": result["median_enrichment"]})
time_shift_summary = pd.DataFrame(time_rows)

baseline_result = parameter_results[("Cumulative anomaly", "S > 2.0")]
baseline_null_values = baseline_result["null"]["median_enrichment"]; baseline_null_values = baseline_null_values[np.isfinite(baseline_null_values)]
baseline_null_low, baseline_null_high = np.nanpercentile(baseline_null_values, [2.5, 97.5])
source_xlsx = OUT_DIR / "FigureS8_PCS_robustness_source_data.xlsx"
with pd.ExcelWriter(source_xlsx, engine="openpyxl") as writer:
    parameter_summary.to_excel(writer, sheet_name="parameter_sensitivity", index=False)
    heatmap_summary.to_excel(writer, sheet_name="threshold_heatmap", index=False)
    time_shift_summary.to_excel(writer, sheet_name="endpoint_perturbation", index=False)
    pd.DataFrame({"item": ["n_events", "dt_seconds", "null_shifts_per_event", "null_replicates", "null_definition"], "value": [len(labels_by_event), DT_SECONDS, NULL_SHIFTS_PER_EVENT, NULL_REPLICATES, "circularly shifted DEC labels; one random non-zero shift per event"]}).to_excel(writer, sheet_name="analysis_settings", index=False)
display(parameter_summary[["family", "parameter", "observed_selected_events", "observed_median_enrichment", "null_enrichment_2.5%", "null_enrichment_97.5%"]])
print(f"Baseline observed selection: {baseline_result['observed']['selected_count']}/{len(labels_by_event)} events")
print(f"Baseline observed median enrichment: {baseline_result['observed']['median_enrichment']:.3f}x")
print(f"Source data: {source_xlsx}")


## Figure S8: parameter and initiation-time robustness

In [ ]:
# =========================== EDIT FIGURE STYLE HERE ===========================
FIGSIZE_MM = (183, 142)
FONT = {"family": "Arial", "base": 7.4, "axis_label": 8.1, "tick": 6.8, "legend": 6.7, "annotation": 6.5, "panel_label": 12.0}
LAYOUT = {"left": 0.080, "right": 0.975, "bottom": 0.105, "top": 0.955, "wspace": 0.33, "hspace": 0.40}
HEATMAP_CMAP = "YlOrRd"; OBSERVED_COLOR = "#B64342"; NULL_COLOR = "#4D4D4D"
ACCENT_COLORS = {"Cumulative anomaly": "#0F4D92", "Nonlinear condition": "#D97732", "Terminal window": "#2A9D8F", "Early reference": "#7B5EA7"}
OUTPUT_FORMATS = ("svg", "pdf", "png", "tiff"); PNG_DPI, TIFF_DPI = 400, 600
mpl.rcParams.update({"font.family": "sans-serif", "font.sans-serif": [FONT["family"], "Helvetica", "DejaVu Sans"], "font.size": FONT["base"], "axes.labelsize": FONT["axis_label"], "xtick.labelsize": FONT["tick"], "ytick.labelsize": FONT["tick"], "legend.fontsize": FONT["legend"], "axes.linewidth": 0.75, "legend.frameon": False, "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none"})

fig_s8 = plt.figure(figsize=(FIGSIZE_MM[0] / 25.4, FIGSIZE_MM[1] / 25.4), facecolor="white")
grid = fig_s8.add_gridspec(2, 2, height_ratios=[1.05, 0.95])
axis_heatmap = fig_s8.add_subplot(grid[0, 0]); axis_interval = fig_s8.add_subplot(grid[0, 1]); axis_shift = fig_s8.add_subplot(grid[1, :])
fig_s8.subplots_adjust(**LAYOUT)

# ----------------------------- a: threshold heatmap -----------------------------
pivot_values = heatmap_summary.pivot(index="z_threshold", columns="c_threshold", values="median_enrichment")
pivot_counts = heatmap_summary.pivot(index="z_threshold", columns="c_threshold", values="selected_events")
heatmap_image = axis_heatmap.imshow(pivot_values.to_numpy(dtype=float), origin="lower", aspect="auto", cmap=HEATMAP_CMAP, vmin=float(np.nanmin(pivot_values)), vmax=float(np.nanmax(pivot_values)))
axis_heatmap.set_xticks(np.arange(len(pivot_values.columns))); axis_heatmap.set_xticklabels([f"R < {value:g}" for value in pivot_values.columns])
axis_heatmap.set_yticks(np.arange(len(pivot_values.index))); axis_heatmap.set_yticklabels([f"S > {value:g}" for value in pivot_values.index])
axis_heatmap.set_xlabel("Nonlinear condition"); axis_heatmap.set_ylabel("Cumulative anomaly threshold")
for row_index, z_threshold in enumerate(pivot_values.index):
    for column_index, c_threshold in enumerate(pivot_values.columns):
        value = float(pivot_values.loc[z_threshold, c_threshold]); count = int(pivot_counts.loc[z_threshold, c_threshold])
        axis_heatmap.text(column_index, row_index, f"{value:.2f}x\n(n={count})", ha="center", va="center", fontsize=FONT["annotation"], color="white" if value > np.nanmedian(pivot_values.to_numpy()) else "0.15")
heatmap_colorbar = fig_s8.colorbar(heatmap_image, ax=axis_heatmap, fraction=0.046, pad=0.04); heatmap_colorbar.set_label("Median terminal enrichment")
axis_heatmap.set_title("Threshold-combination response", loc="left", fontsize=FONT["axis_label"]); axis_heatmap.text(-0.18, 1.12, "a", transform=axis_heatmap.transAxes, fontsize=FONT["panel_label"], fontweight="bold")

# ---------------------- b: observed values and null intervals -------------------
interval_order = [("Cumulative anomaly", "S > 1.5"), ("Cumulative anomaly", "S > 2.0"), ("Cumulative anomaly", "S > 2.5"), ("Nonlinear condition", "R < 0.8"), ("Nonlinear condition", "R < 1.0"), ("Nonlinear condition", "R < 1.2"), ("Terminal window", "0.5 h"), ("Terminal window", "1 h"), ("Terminal window", "2 h"), ("Early reference", "6 h"), ("Early reference", "9 h"), ("Early reference", "12 h")]
interval_plot = parameter_summary.set_index(["family", "parameter"]).loc[interval_order].reset_index(); x_interval = np.arange(len(interval_plot), dtype=float)
for index, row in interval_plot.iterrows():
    axis_interval.vlines(x_interval[index], row["null_enrichment_2.5%"], row["null_enrichment_97.5%"], color=NULL_COLOR, linewidth=1.2, zorder=2)
    axis_interval.scatter(x_interval[index], row["observed_median_enrichment"], s=28, color=ACCENT_COLORS[row["family"]], edgecolor="white", linewidth=0.45, zorder=3)
for boundary in [2.5, 5.5, 8.5]: axis_interval.axvline(boundary, color="0.75", linewidth=0.6, linestyle=":")
axis_interval.set_xticks(x_interval); axis_interval.set_xticklabels(interval_plot["parameter"], rotation=55, ha="right"); axis_interval.set_ylabel("Median terminal enrichment")
axis_interval.set_title("Observed value vs circular-shift null", loc="left", fontsize=FONT["axis_label"]); axis_interval.text(-0.16, 1.12, "b", transform=axis_interval.transAxes, fontsize=FONT["panel_label"], fontweight="bold")
axis_interval.grid(axis="y", color="0.75", linewidth=0.45, alpha=0.25, zorder=0); axis_interval.spines["top"].set_visible(False); axis_interval.spines["right"].set_visible(False)
for center, label in [(1, "S"), (4, "R"), (7, "terminal"), (10, "early")]: axis_interval.text(center, -0.34, label, transform=axis_interval.get_xaxis_transform(), ha="center", va="top", fontsize=FONT["annotation"], color="0.35")
axis_interval.plot([], [], color=NULL_COLOR, linewidth=1.2, label="Null 95% interval"); axis_interval.scatter([], [], s=28, color=OBSERVED_COLOR, label="Observed median"); axis_interval.legend(loc="upper right", handlelength=1.3, labelspacing=0.45)

# ---------------------------- c: endpoint perturbation --------------------------
axis_shift.fill_between(offsets_min, baseline_null_low, baseline_null_high, color=NULL_COLOR, alpha=0.16, label="Baseline circular-shift null 95% interval")
axis_shift.axhline(baseline_null_low, color=NULL_COLOR, linewidth=0.55, linestyle="--", alpha=0.65); axis_shift.axhline(baseline_null_high, color=NULL_COLOR, linewidth=0.55, linestyle="--", alpha=0.65)
axis_shift.plot(time_shift_summary["offset_min"], time_shift_summary["median_enrichment"], color=OBSERVED_COLOR, marker="o", markersize=4.5, linewidth=1.25, label="Observed enrichment")
for _, row in time_shift_summary.iterrows():
    if np.isfinite(row["median_enrichment"]): axis_shift.text(row["offset_min"], row["median_enrichment"] + 0.10, f"n={int(row['selected_events'])}", ha="center", va="bottom", fontsize=FONT["annotation"], color="0.35")
axis_shift.axvline(0, color="0.35", linewidth=0.65, linestyle=":"); axis_shift.set_xticks(offsets_min); axis_shift.set_xlabel("Circular label shift relative to catalogued initiation time (min)"); axis_shift.set_ylabel("Median terminal enrichment")
axis_shift.set_title("Robustness to initiation-time perturbation", loc="left", fontsize=FONT["axis_label"]); axis_shift.text(-0.075, 1.12, "c", transform=axis_shift.transAxes, fontsize=FONT["panel_label"], fontweight="bold")
axis_shift.grid(axis="y", color="0.75", linewidth=0.45, alpha=0.25, zorder=0); axis_shift.spines["top"].set_visible(False); axis_shift.spines["right"].set_visible(False); axis_shift.legend(loc="upper left", ncol=2, handlelength=1.5, columnspacing=1.2)

combined_stem = OUT_DIR / "FigureS8_PCS_parameter_and_time_robustness"; combined_paths = []
for extension in OUTPUT_FORMATS:
    output_path = combined_stem.with_suffix(f".{extension}"); save_options = {"bbox_inches": "tight", "facecolor": "white", "pad_inches": 0.04}
    if extension == "png": save_options["dpi"] = PNG_DPI
    elif extension == "tiff": save_options["dpi"] = TIFF_DPI; save_options["pil_kwargs"] = {"compression": "tiff_lzw"}
    fig_s8.savefig(output_path, **save_options); combined_paths.append(output_path)
display(fig_s8); plt.close(fig_s8); print("Figure S8 exports:", [item.name for item in combined_paths])


## Interpretation notes

Heatmap values are observed median terminal enrichments with selected-event counts in parentheses. Point–interval marks compare observed values with circular-shift null 95% intervals. The endpoint curve uses the baseline PCS configuration; the shaded band is its study-level null interval. These are robustness diagnostics rather than prospective detection-performance estimates.